In [1]:
# @title
# ==========================================
# CONFIGURACIÓN DEL ENTORNO (EJECUTAR PRIMERO)
# ==========================================
import pandas as pd
import sqlite3

# 1) URLs de archivos crudos (Raw) en GitHub
urls = {
    'clientes': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/clientes.csv",
    'ventas': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/ventas.csv",
    'trabajadores': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/trabajadores.csv",
    'cargos': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/cargos.csv",
    'areas': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/areas.csv"
}

print("Preparando base de datos 'techstore.db'.. .")

# 2) Crear la conexión y la base de datos
conn = sqlite3.connect('techstore.db')

# 3) Leer cada CSV e insertarlo como tabla en la base de datos
for tabla, url in urls.items():
    try:
        df = pd.read_csv(url, sep=None, engine='python')
        df.to_sql(tabla, conn, index=False, if_exists='replace')
        print(f"Tabla '{tabla}' cargada exitosamente.")
    except Exception as e:
        print(f"Error al cargar la tabla '{tabla}': {e}")

conn.close()
print("¡Entorno listo! La base de datos relacional está disponible para ser consultada.")

Preparando base de datos 'techstore.db'.. .
Tabla 'clientes' cargada exitosamente.
Tabla 'ventas' cargada exitosamente.
Tabla 'trabajadores' cargada exitosamente.
Tabla 'cargos' cargada exitosamente.
Tabla 'areas' cargada exitosamente.
¡Entorno listo! La base de datos relacional está disponible para ser consultada.


In [3]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('techstore.db')

print("Conexión exitosa")

Conexión exitosa


In [5]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

tablas = pd.read_sql(query,conn)
tablas

,name
0,clientes
1,ventas
2,trabajadores
3,cargos
4,areas


In [7]:
clientes = pd.read_sql("SELECT * FROM clientes", conn)
ventas = pd.read_sql("SELECT * FROM ventas", conn)
trabajadores = pd.read_sql("SELECT * FROM trabajadores", conn)
cargos = pd.read_sql("SELECT * FROM cargos", conn)
areas = pd.read_sql("SELECT * FROM areas", conn)

print("Tablas cargadas")

Tablas cargadas


In [8]:
print("Clientes: ", clientes.shape)
print("Ventas: ", ventas.shape)
print("Trabajadores: ", trabajadores.shape)
print("Cargos: ", cargos.shape)
print("Areas: ", areas.shape)


Clientes:  (252, 8)
Ventas:  (1200, 7)
Trabajadores:  (35, 5)
Cargos:  (10, 4)
Areas:  (5, 3)


In [11]:
clientes.head()

,id,rut,nombre,segundo_nombre,apellido,segundo_apellido,edad,fecha_nacimiento
0,1,12154051-7,Carlos,Javier,Medina,Rojas,55.0,13-04-1971
1,2,13553384-K,Bárbara,Carolina,Martínez,Valenzuela,29.0,17-12-1997
2,3,17350099-8,Rodrigo,Alejandro,Vergara,Araya,33.0,02-07-1993
3,4,10352896-8,Javiera,Valentina,Silva,Rojas,36.0,01-05-1990
4,5,13841005-6,Matías,Manuel,González,Rojas,61.0,13-04-1965


In [12]:
ventas.head()

,id,id_cliente,id_vendedor,dte,total_dte,estado,descripcion
0,1,191,15,10001,650631,Entregado,"2x Disco Duro Externo 2TB, 3x Mouse Ergonómico..."
1,2,109,3,10004,903198,Entregado,"2x Escáner de Documentos Portátil, 1x Mouse Er..."
2,3,76,10,10007,231213,Entregado,"2x Escáner de Documentos Portátil, 2x Memoria ..."
3,4,158,14,10009,752982,Sin entregar,"3x Licencia Software Antivirus 1 Año, 3x Lapto..."
4,5,132,6,10011,36388,Entregado,3x Mouse Ergonómico Inalámbrico


In [18]:
query = """
SELECT
  id_cliente, COUNT(*) AS cantidad_compras
FROM ventas
GROUP BY id_cliente
ORDER BY cantidad_compras DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,id_cliente,cantidad_compras
0,147,11
1,148,10
2,84,10
3,82,10
4,81,10
5,73,10
6,48,10
7,30,10
8,224,9
9,180,9


In [19]:
query = """
SELECT *
FROM ventas
WHERE id_cliente = 82
ORDER BY dte;
"""

pd.read_sql(query, conn)

,id,id_cliente,id_vendedor,dte,total_dte,estado,descripcion
0,27,82,12,10061,308403,Entregado,"1x Mouse Ergonómico Inalámbrico, 2x Disco Duro..."
1,222,82,6,10470,749963,Entregado,"1x Teclado Mecánico RGB, 2x Licencia Software ..."
2,230,82,9,10487,740360,Entregado,2x Monitor LED 24' Full HD
3,457,82,33,10948,913203,Entregado,"3x Silla de Escritorio Ergonómica, 2x Teclado ..."
4,507,82,3,11048,267565,Entregado,2x Mouse Ergonómico Inalámbrico
5,677,82,12,11392,909252,Sin entregar,"3x Disco Duro Externo 2TB, 1x Silla de Escrito..."
6,758,82,33,11544,894435,Entregado,"1x Silla de Escritorio Ergonómica, 2x Licencia..."
7,763,82,14,11555,749816,Sin entregar,1x Impresora Láser Multifuncional
8,1021,82,12,12060,656565,Entregado,"3x Auriculares con Micrófono USB, 3x Licencia ..."
9,1129,82,5,12277,935561,Entregado,"3x Silla de Escritorio Ergonómica, 2x Hub USB-..."


**Contexto de la Prueba: E-Commerce "TechStore"**

*Escenario: Eres el nuevo analista de datos de TechStore. El equipo ha notado una fluctuación en los ingresos durante el último trimestre y necesita entender qué está pasando.*

**Instrucciones Iniciales:**

1.   La prueba tiene una duración estimada de 45 mins, pero tendrás 72 horas para enviar tus resultados.
2.	Luego de terminar tus ejercicios envíame el enlace en compartir para tu entorno donde fue desarrollado.
3.	Cualquier archivo externo generado debes subirlo y compartir el enlace en una nueva hoja de google colab, en caso de usar hojas de cálculo de Google debes copiar el enlace en otra hoja de google colab, recuerda titular dicho enlace para saber que iré a mirar.
4.	Dentro de la prueba, ve a Archivo > Guardar una copia en Drive para tener tu propia versión editable de esta prueba.
5.	Ejecuta la primera celda de este cuaderno. Esto generará automáticamente una base de datos SQLite llamada techstore.db en este entorno.
6.	El diccionario de datos disponible en la base de datos es:
*   clientes: id, rut, nombre, segundo_nombre, apellido, segundo_apellido, edad, fecha_nacimiento
*   ventas: id, id_cliente, id_vendedor, dte, total_dte, estado, descripcion
*   trabajadores: id, id_cargo, nombre, apellido, rut
*   cargos: id, nombre_cargo, descripcion, id_area
*   areas: id, nombre_area, descripcion

*disclaimer: Puedes usar IA, sin embargo, revisaré cada linea de código para validar la efectividad y también la totalidad del cumplimiento de las respuestas.*

**Parte 1: Conexión y Extracción (SQL & Python)**

*Conéctate a la base de datos techstore.db usando Python (por ejemplo, con la librería sqlite3 o sqlalchemy). Luego, resuelve las siguientes consultas ejecutando SQL puro dentro de tu código Python:*
1.   **Ventas Totales:** Escribe una consulta para calcular el ingreso total generado por órdenes con estado "Entregado" y el % del total de ventas que representa.
2.   **Top Clientes:** Encuentra los 5 usuarios que han gastado más dinero históricamente, mostrando su id, rut y el total_dte gastado.
3.   **Retención:** Calcula el porcentaje de usuarios que realizaron una segunda compra dentro de los 200 dte siguientes (si alguien compró el dte 11721 y compró nuevamente antes ó justamente en el dte 11921 es considerado).

In [16]:
#ventas totales
query = """
SELECT
  SUM(CASE
        WHEN estado = 'Entregado'
        THEN total_dte
        ELSE 0
      END) AS ventas_entregadas,
  SUM(total_dte) AS ventas_totales,

  ROUND(
        (SUM(CASE
                WHEN estado = 'Entregado'
                THEN total_dte
                ELSE 0
              END) * 100.0
        ) / SUM(total_dte)
        ,2) AS porcentaje
FROM ventas;
"""

resultado = pd.read_sql(query, conn)

resultado

,ventas_entregadas,ventas_totales,porcentaje
0,477191018,588673484,81.06


"Del total facturado por TechStore ($588.673.484), el 81,06% corresponde a ventas efectivamente entregadas. El 18,94% restante corresponde a pedidos en otros estados, lo que podría representar ventas pendientes de despacho o posibles pérdidas de ingresos si no se concretan."

In [53]:
#Usuarios que han gastado mas dinero
query = """
SELECT
  c.id, c.rut, c.nombre, c.apellido,
  SUM(v.total_dte) AS gasto_total
FROM clientes c
INNER JOIN ventas v
  ON c.id = v.id_cliente
GROUP BY
  c.id, c.rut
ORDER BY gasto_total DESC
LIMIT 5;
"""

top_clientes = pd.read_sql(query, conn)

top_clientes

,id,rut,nombre,apellido,gasto_total
0,82,21866669-8,Sofía,Sepúlveda,7125123
1,81,10212267-4,Natalia,Valenzuela,5663000
2,94,11686548-3,Javier,Donoso,5621201
3,145,14605016-6,Paulina,Medina,5604037
4,48,19301497-6,Patricia,Herrera,5106230


"Los cinco clientes con mayor gasto histórico registran compras acumuladas entre 5 y 7 millones. Esto evidencia que una pequeña cantidad de clientes concentra una parte importante de los ingresos, por lo que podrían considerarse estrategias de fidelización o programas de beneficios para este segmento."

In [21]:
#Ver cuantos clientes han comprado
query = """
SELECT
  id_cliente,
  MIN(dte) AS primer_dte
FROM ventas
GROUP BY id_cliente
ORDER BY primer_dte;
"""

primeras_compras = pd.read_sql(query, conn)
primeras_compras.head()

print("Clientes con compras: ", primeras_compras.shape[0])

Clientes con compras:  248


In [22]:
#Porcentaje de retención
query = """
WITH primeras_compras AS (
  SELECT
    id_cliente,
    MIN(dte) AS primer_dte
  FROM ventas
  GROUP BY id_cliente
)

SELECT
  COUNT(DISTINCT pc.id_cliente) AS clientes_retenidos,
  (
    SELECT COUNT(DISTINCT id_cliente)
    FROM ventas
  ) AS total_clientes,
  ROUND(
    COUNT(DISTINCT pc.id_cliente) * 100.0 /
    (
      SELECT COUNT(DISTINCT id_cliente)
      FROM ventas
    ),
    2
  ) AS porcentaje_retencion
FROM primeras_compras pc
JOIN ventas v
  ON pc.id_cliente = v.id_cliente
WHERE
  v.dte > pc.primer_dte
  AND v.dte <= pc.primer_dte + 200;
"""

retencion = pd.read_sql(query, conn)

retencion

,clientes_retenidos,total_clientes,porcentaje_retencion
0,76,248,30.65


"El 30,65% de los clientes realizó una segunda compra dentro de los 200 DTE posteriores a su primera compra. Esto indica que aproximadamente 3 de cada 10 clientes vuelven a comprar en un período relativamente corto, mientras que el 69,35% restante no presenta una recompra temprana, representando una oportunidad para estrategias de fidelización y remarketing."

**Parte 2: Análisis Exploratorio (Python)**

*Trae las tablas necesarias a DataFrames de Pandas y resuelve:*

1.	**Limpieza:** Identifica y muestra los valores nulos o duplicados en la tabla de clientes. Comenta tu criterio.
2.	**Métricas:** Identifica el top 10 de total_dte más altos, el top 3 de clientes por volumen de transacciones (cantidad de compras), y el top 3 de vendedores (id_vendedor) con mayor monto acumulado, solo considera vendedores, si hay otros cargos comerciales que han realizado ventas debes ignorarlos.
3.	**Volumen:** Crea un DataFrame del top 10 de productos mas vendidos ordenado de mayor a menor que muestre el id de la venta, nombre del producto y unidades vendidas (deberás procesar la columna descripcion).


In [23]:
#Identificar valores nulos
clientes.isnull().sum()

,0
id,0
rut,0
nombre,0
segundo_nombre,0
apellido,0
segundo_apellido,0
edad,4
fecha_nacimiento,0


In [30]:
#Identificar valores duplicados
clientes.duplicated().sum()
clientes[clientes.duplicated()]

,id,rut,nombre,segundo_nombre,apellido,segundo_apellido,edad,fecha_nacimiento


"Se realizó una revisión de calidad de datos sobre la tabla clientes. Se identificaron 4 registros con valores nulos en la columna edad, mientras que el resto de las columnas no presenta datos faltantes. Respecto a registros duplicados, no se encontraron filas completamente repetidas. Dado que la edad podría derivarse a partir de la fecha de nacimiento, no se eliminaron registros y se recomienda evaluar una imputación o recálculo de dicho atributo según las necesidades del negocio."

In [31]:
#Top 10 ventas
top10_ventas = ventas.nlargest(10, 'total_dte')

top10_ventas[['id', 'id_cliente', 'total_dte', 'estado']]

,id,id_cliente,total_dte,estado
320,321,75,999825,Entregado
483,484,40,998087,Entregado
627,628,45,996482,Entregado
661,662,182,996396,Entregado
458,459,144,996276,Sin entregar
114,115,201,995217,Entregado
734,735,110,995031,Entregado
442,443,106,994428,Entregado
178,179,163,994032,Entregado
207,208,72,992339,Entregado


"De las 10 ventas de mayor monto, 9 corresponden a ventas entregadas y 1 se encuentra sin entregar. Esto sugiere que las ventas de alto valor presentan una alta tasa de concretización."

In [45]:
#Top 3 Clientes
top3_clientes = (
    ventas.groupby('id_cliente')
    .size()
    .reset_index(name='cantidad_compras')
    .sort_values('cantidad_compras', ascending=False)
    .head(3)
)

top3_clientes = top3_clientes.merge(
    clientes,
    left_on='id_cliente',
    right_on='id'
)

top3_clientes['nombre_completo'] = (
    top3_clientes['nombre']
    + ' '
    + top3_clientes['apellido']
)

top3_clientes[
    [
        'id_cliente',
        'nombre_completo',
        'cantidad_compras'
    ]
]

,id_cliente,nombre_completo,cantidad_compras
0,147,Nicolás Silva,11
1,48,Patricia Herrera,10
2,84,Alejandra Medina,10


"Se identificó el Top 3 de clientes con mayor volumen de transacciones considerando la cantidad total de compras registradas. El cliente Nicolás Silva lidera el ranking con 11 compras, seguido por Patricia Herrera y Alejandra Medina con 10 compras cada una. Estos clientes representan una alta recurrencia de compra y podrían ser considerados para estrategias de fidelización, programas de beneficios o campañas de marketing dirigidas a clientes frecuentes."

In [41]:
#Revisar cargos
cargos

,id,nombre_cargo,descripcion,id_area
0,1,Gerente Comercial,Liderar las estrategias de venta y cumplimient...,1
1,2,Ejecutivo de Ventas Senior,Venta directa y gestión de grandes cuentas de ...,1
2,3,Ejecutivo de Ventas Junior,Atención de prospectos y cierre de ventas de c...,1
3,4,Jefe de Logística,Coordinar la cadena de suministro y despacho a...,2
4,5,Operario de Bodega,"Preparación de pedidos, recepción de mercaderí...",2
5,6,Ingeniero de Soporte TI,"Resolver incidencias de hardware, software y r...",3
6,7,Desarrollador Full Stack,Mantenimiento y desarrollo de las plataformas ...,3
7,8,Analista de RRHH,"Procesamiento de remuneraciones, contratos y g...",4
8,9,Contador General,"Declaraciones de impuestos, balances comercial...",5
9,10,Analista Financiero,"Control de presupuestos, flujo de caja y relac...",5


In [42]:
#Top 3 Vendedores
ventas_vendedores = (
    ventas.merge(
        trabajadores,
        left_on='id_vendedor',
        right_on='id',
        suffixes=('_venta', '_trabajador')
    ).merge(
        cargos,
        left_on='id_cargo',
        right_on='id',
        suffixes=('', '_cargo')
    )
)

#filtrar solo vendedores
ventas_vendedores = ventas_vendedores[
    ventas_vendedores['nombre_cargo'].isin([
        'Ejecutivo de Ventas Senior',
        'Ejecutivo de Ventas Junior'
    ])
]

top3_vendedores = (
    ventas_vendedores.groupby(
        ['id_vendedor', 'nombre', 'apellido', 'nombre_cargo']
    )['total_dte']
    .sum()
    .reset_index()
    .sort_values('total_dte', ascending=False)
    .head(3)
)

top3_vendedores

,id_vendedor,nombre,apellido,nombre_cargo,total_dte
4,5,Felipe,Sepúlveda,Ejecutivo de Ventas Senior,40264872
12,13,Manuel,Cortés,Ejecutivo de Ventas Senior,38625479
6,7,Andrea,Martínez,Ejecutivo de Ventas Senior,36945022


"Para identificar a los vendedores con mejor desempeño, se consideraron únicamente los trabajadores con cargos de venta (Ejecutivo de Ventas Senior y Ejecutivo de Ventas Junior), excluyendo otros cargos comerciales y administrativos. El análisis muestra que Felipe Sepúlveda lidera el ranking con un monto acumulado de ventas de 40.264.872, seguido por Manuel Cortés con 38.625.479 y Andrea Martínez con 36.945.022. Estos resultados permiten identificar a los ejecutivos con mayor contribución a los ingresos de la compañía y podrían utilizarse como referencia para analizar buenas prácticas comerciales, definir incentivos o evaluar oportunidades de capacitación para el equipo de ventas."


In [46]:
#Revisar campo descripción
ventas['descripcion'].sample(15, random_state=42)

,descripcion
1178,2x Disco Duro Externo 2TB
865,"1x Mouse Ergonómico Inalámbrico, 1x Impresora ..."
101,2x Disco Duro Externo 2TB
439,"3x Impresora Láser Multifuncional, 2x Escáner ..."
58,"1x Monitor LED 24' Full HD, 3x Disco Duro Exte..."
1120,"1x Licencia Software Antivirus 1 Año, 1x Auric..."
323,"2x Impresora Láser Multifuncional, 3x Mouse Er..."
974,"3x Mouse Ergonómico Inalámbrico, 1x Impresora ..."
411,"2x Silla de Escritorio Ergonómica, 2x Mouse Er..."
855,"2x Silla de Escritorio Ergonómica, 2x Escáner ..."


In [48]:
#Lista con todos los productos de todas las ventas
import re
productos = []
for _, fila in ventas.iterrows():
  id_venta = fila['id']
  descripcion = fila['descripcion']

  items =  descripcion.split(',')

  for item in items:
    item = item.strip()
    match = re.match(r'(\d+)x\s+(.*)', item)
    if match:
      cantidad = int(match.group(1))
      producto = match.group(2)

      productos.append({
                'id_venta': id_venta,
                'producto': producto,
                'unidades': cantidad
            })

productos_df = pd.DataFrame(productos)
productos_df.head()

,id_venta,producto,unidades
0,1,Disco Duro Externo 2TB,2
1,1,Mouse Ergonómico Inalámbrico,3
2,1,Silla de Escritorio Ergonómica,2
3,1,Escáner de Documentos Portátil,1
4,2,Escáner de Documentos Portátil,2


In [49]:
#Obtener top 10

top10_productos=(
    productos_df
    .groupby('producto')
    .agg({
        'unidades':'sum',
        'id_venta': 'count'
    })
    .reset_index()
    .rename(columns={
        'id_venta': 'cantidad_ventas'
    })
    .sort_values('unidades', ascending=False)
    .head(10)
)

top10_productos

,producto,unidades,cantidad_ventas
2,Escáner de Documentos Portátil,551,269
9,Mouse Ergonómico Inalámbrico,532,265
5,Laptop Corporate 14',504,253
3,Hub USB-C 8 en 1,502,247
7,Memoria RAM 16GB DDR4,492,243
1,Disco Duro Externo 2TB,492,243
6,Licencia Software Antivirus 1 Año,486,255
10,Silla de Escritorio Ergonómica,479,242
11,Teclado Mecánico RGB,475,233
0,Auriculares con Micrófono USB,471,249


"Para identificar los productos más vendidos fue necesario procesar la columna "descripcion" de la tabla ventas, extrayendo la cantidad y el nombre de cada producto mediante expresiones regulares. Posteriormente, se consolidaron las unidades vendidas por producto para obtener el ranking de demanda. Dado que la base de datos no contiene una tabla de productos ni identificadores únicos asociados a ellos, el análisis se realizó utilizando el nombre del producto como identificador lógico. Adicionalmente, se incorporó la cantidad de ventas en las que aparece cada producto para complementar el análisis de popularidad."


**Parte 3: Lógica de Negocio (Hojas de Cálculo / Excel)**

*Para esta sección, deberás ingeniártelas para extraer/descargar los datos de este entorno (ej. generando archivos CSV desde tus DataFrames o desde la DB) e importarlos a Google Sheets o Excel, puedes utilizar la tecnología que estimes conveniente, lo importante es resolver en Excel.*

1.	**Consolidación**: En una hoja nueva, toma una muestra de las primeras 100 ventas según el dte asociado (a menor folio, el documento es más antiguo). Usa fórmulas de búsqueda (BUSCARV, BUSCARX o INDICE/COINCIDIR) para traer el nombre y apellido del cliente y el nombre_cargo del vendedor asociado a esa venta.
2.	**Tabla Dinámica:** Crea una tabla dinámica que muestre el monto total vendido por cada área de la empresa (nombre_area), filtrando solo el estado "Entregado".
3.	**Segmentación:** En tu hoja de ventas, crea la columna "Categoría de Ticket". Usa funciones lógicas para clasificar la venta como "Ticket Alto" (si el total_dte supera el promedio general) o "Ticket Bajo" (si es igual o inferior).


In [54]:
#Exportar archivos
clientes.to_csv('clientes.csv', index=False)
ventas.to_csv('ventas.csv', index=False)
trabajadores.to_csv('trabajadores.csv', index=False)
cargos.to_csv('cargos.csv', index=False)
areas.to_csv('areas.csv', index=False)

print("Archivos exportados correctamente")

Archivos exportados correctamente


In [57]:
#Exportar solo las primeras 100 ventas
primeras_100_ventas = (
    ventas
    .sort_values('dte')
    .head(100)
)

primeras_100_ventas.to_csv(
    'primeras_100_ventas.csv',
    index=False
)

primeras_100_ventas.head()

,id,id_cliente,id_vendedor,dte,total_dte,estado,descripcion
0,1,191,15,10001,650631,Entregado,"2x Disco Duro Externo 2TB, 3x Mouse Ergonómico..."
1,2,109,3,10004,903198,Entregado,"2x Escáner de Documentos Portátil, 1x Mouse Er..."
2,3,76,10,10007,231213,Entregado,"2x Escáner de Documentos Portátil, 2x Memoria ..."
3,4,158,14,10009,752982,Sin entregar,"3x Licencia Software Antivirus 1 Año, 3x Lapto..."
4,5,132,6,10011,36388,Entregado,3x Mouse Ergonómico Inalámbrico


**Opcional (Puntos extra):**

**Presentación al Negocio (Dashboard + Insights)**

1.	**Visualización**: Conecta tus datos exportados a Power BI, Looker o Tableau. Crea un dashboard de una página con:


*   KPI de Ingresos Totales y Ticket Promedio.
*   Gráfico comparativo de rendimiento por categoría/producto.
*   Filtros interactivos.
2.	**Estrategia**: En un documento breve o en celdas de texto al final de este cuaderno, responde:

*   ¿Cuáles fueron tus hallazgos respecto a la fluctuación de ingresos?
*   Propón dos recomendaciones accionables para marketing/ventas.

**Control de versiones (Github):**

1.	**Creación del repositorio:** Entregar los scripts (SQL/Python) subidos a un repositorio propio de GitHub.


# Hallazgos
Durante el análisis se observó que el 81,06% de los ingresos corresponde a ventas entregadas, mientras que el 18,94% restante se encuentra en estados distintos a entregado. La tasa de retención obtenida fue de 30,65%, indicando que aproximadamente 3 de cada 10 clientes realizan una recompra dentro de los 200 DTE posteriores a su primera compra. Además, los productos con mayor demanda fueron Escáner de Documentos Portátil y Mouse Ergonómico Inalámbrico, ambos superando las 500 unidades vendidas. En cuanto al desempeño comercial, Felipe Sepúlveda fue el vendedor con mayor monto acumulado de ventas.

# Recomendaciones
1.- Implementar campañas de fidelización dirigidas a clientes que realizaron una primera compra pero no efectuaron una recompra dentro de los 200 DTE siguientes.


2.- Potenciar promociones y asegurar stock de los productos más vendidos para maximizar ingresos y evitar quiebres de inventario.

########################################
# Entregables

Google Sheets
https://docs.google.com/spreadsheets/d/15TT2pTfWqXbg3smS5lASRFQrrhCka3rne136-O_eS88/edit?usp=sharing

Dashboard Looker Studio
https://datastudio.google.com/reporting/56ca9b5a-5444-4179-9a52-566cb94c2bb0

Repositorio GitHub
https://github.com/Marcelgutierr/prueba-tecnica-analista-datos-alca